# Genetic Algorithm — example with several classes (ordinal)

This notebook uses `multiclass_data.xlsx`, which has **5 classes**: D, G, M, P and S.

With several classes the program uses `ordinal_fitness`. It needs the classes to have an **order** (from lowest to highest), because the model gives each sample a score and splits that score into ranges, one range per class.

Since the real order of the classes is not known yet, the notebook **tries every possible order** and uses the one with the best fitness. Once you know the right order, just put it in the `CLASS_ORDER` variable (section 3).

## 0. Imports

In [1]:
import sys
sys.path.insert(0, "../../src")  # folder that holds the genetic_classifier package

import itertools
import time

import numpy as np

from genetic_classifier.reading import read_excel
from genetic_classifier.creation import create_chromosomes
from genetic_classifier.fitness import ordinal_fitness
from genetic_classifier import train, predict

np.random.seed(0)  # seed: running again gives the same random draws

## 1. Reading the data

In [2]:
X, y_letters = read_excel("multiclass_data.xlsx")
X = X.astype(float)

classes = sorted(set(y_letters))

print("X (n_samples, n_features):", X.shape)
print("classes:", classes)
for c in classes:
    print(f"  {c}: {sum(y_letters == c)} samples")

X (n_samples, n_features): (261, 30)
classes: ['D', 'G', 'M', 'P', 'S']
  D: 37 samples
  G: 61 samples
  M: 39 samples
  P: 64 samples
  S: 60 samples


## 2. Splitting train and test

80% for training and 20% for testing. Choosing the order of the classes (section 3) uses **only the training set**, so the test set stays data the model has never seen.

In [3]:
sample_order = np.random.permutation(len(y_letters))  # numbers 0 to n-1 shuffled
cut = int(0.8 * len(y_letters))

X_train, y_train_letters = X[sample_order[:cut]], y_letters[sample_order[:cut]]
X_test, y_test_letters = X[sample_order[cut:]], y_letters[sample_order[cut:]]

print("train:", X_train.shape, "| test:", X_test.shape)

train: (208, 30) | test: (53, 30)


## 3. How the ordinal model works

Each chromosome computes a score for the sample: `score = bias + sum(weight × feature)`.

The **thresholds** (cut points) split the score into ranges. With 5 classes we need 4 cuts:

```
        class 0  |  class 1  |  class 2  |  class 3  |  class 4
  ... ---------- -1.5 ------ -0.5 ------ 0.5 ------- 1.5 ---------- ...
```

Internally, the program uses `np.digitize(score, thresholds)`, which just tells which range the score fell into (0, 1, 2, ...). Example:

In [4]:
THRESHOLDS = np.array([-1.5, -0.5, 0.5, 1.5])

scores = np.array([-3.0, -1.0, 0.0, 1.0, 2.7])
print("scores:", scores)
print("ranges:", np.digitize(scores, THRESHOLDS))

scores: [-3.  -1.   0.   1.   2.7]
ranges: [0 1 2 3 4]


The fitness is the **product of the recall of each class**: if the model gets every sample of a class wrong, the fitness becomes 0. This forces the model to get at least some of every class right.

### Choosing the order of the classes

If you know the real order, put it here, from lowest to highest, for example `CLASS_ORDER = ["D", "M", "S", "G", "P"]`.
Leaving it as `None`, the next cell tries every order.

In [5]:
CLASS_ORDER = None

### Searching for the best order

`itertools.permutations` (from Python itself) generates every possible order of the 5 classes (5! = 120).
An order and the same order reversed give the same result (it only flips the sign of the weights), so we test only 60.

For each order, it runs the algorithm with 2 different seeds and averages the fitness on the **training set**. It takes a few seconds.

In [6]:
def to_numbers(y_letters, order):
    """Replace each letter by its position in the order (e.g. order D,M,S -> D=0, M=1, S=2)."""
    return np.array([order.index(letter) for letter in y_letters])


if CLASS_ORDER is None:
    start = time.time()
    results = []

    for order in itertools.permutations(classes):
        if order[0] > order[-1]:  # skip the reversed order
            continue

        y_train = to_numbers(y_train_letters, list(order))
        fitnesses = []
        for seed in [0, 1]:
            np.random.seed(seed)
            m = train(X_train, y_train, thresholds=THRESHOLDS, n_chromosomes=6, n_mutated_genes=1,
                      n_offspring=3, n_replacements=2, n_generations=500, target_fitness=1.0, verbose=False)
            fitnesses.append(m["fitness"])

        results.append((sum(fitnesses) / len(fitnesses), list(order)))

    results.sort(reverse=True)  # highest fitness first
    print(f"tested {len(results)} orders in {time.time() - start:.1f}s\n")
    print("5 best orders:")
    for mean_fitness, order in results[:5]:
        print(f"  {' < '.join(order)}   mean fitness = {mean_fitness:.4f}")
    print("\nworst order:")
    print(f"  {' < '.join(results[-1][1])}   mean fitness = {results[-1][0]:.4f}")

    CLASS_ORDER = results[0][1]

print("\norder used:", " < ".join(CLASS_ORDER))

tested 60 orders in 14.0s

5 best orders:
  P < D < G < M < S   mean fitness = 0.3737
  D < P < G < M < S   mean fitness = 0.3678
  D < G < P < M < S   mean fitness = 0.3532
  D < M < S < G < P   mean fitness = 0.3463
  G < P < M < D < S   mean fitness = 0.3409

worst order:
  G < P < S < D < M   mean fitness = 0.0518

order used: P < D < G < M < S


> The best orders usually have similar fitness, so the winner may change with other seeds. The gap between the best and the worst is much larger, so the search already rules out the bad orders.

## 4. Converting the classes to numbers

In [7]:
y_train = to_numbers(y_train_letters, CLASS_ORDER)
y_test = to_numbers(y_test_letters, CLASS_ORDER)

for number, letter in enumerate(CLASS_ORDER):
    print(f"{letter} -> {number}")

P -> 0
D -> 1
G -> 2
M -> 3
S -> 4


## 5. Looking at the fitness of a random population

Before training: random chromosomes almost always have fitness 0 or very low, because it is hard to get some of **all** 5 classes right by luck.

In [8]:
np.random.seed(0)
population = create_chromosomes(n_chromosomes=6, n_genes=X_train.shape[1] + 1)
fitnesses = ordinal_fitness(population, X_train, y_train, THRESHOLDS)

for i, f in enumerate(fitnesses):
    print(f"chromosome {i}: fitness = {f:.6f}")

chromosome 0: fitness = 0.000000
chromosome 1: fitness = 0.000000
chromosome 2: fitness = 0.000000
chromosome 3: fitness = 0.000000
chromosome 4: fitness = 0.001290
chromosome 5: fitness = 0.000226


## 6. Training with `train`

`target_fitness=1.0` makes it run every generation (with 5 classes, 1.0 would mean getting everything right).
Use `verbose=True` to see the best fitness of every generation.

In [9]:
np.random.seed(0)
start = time.time()

model = train(
    X_train, y_train,
    n_chromosomes=6,
    n_mutated_genes=1,
    n_offspring=3,
    n_replacements=2,
    n_generations=2000,
    target_fitness=1.0,
    thresholds=THRESHOLDS,
    verbose=False,
)

print(f"time: {time.time() - start:.1f}s")
print("best fitness:", round(model["fitness"], 4))
print("generations:", model["generation"])

time: 0.5s
best fitness: 0.295
generations: 2000


## 7. Testing with `predict`

In [10]:
y_pred = predict(model, X_test)

hits = sum(y_pred == y_test)
print(f"test accuracy: {hits}/{len(y_test)} = {hits / len(y_test):.2%}")

# for comparison: always guessing the most common class in training
most_common = max(range(len(CLASS_ORDER)), key=lambda k: sum(y_train == k))
guess = sum(y_test == most_common) / len(y_test)
print(f"always guessing '{CLASS_ORDER[most_common]}': {guess:.2%}")

print("\nrecall per class (how much of each class it got right):")
for k, letter in enumerate(CLASS_ORDER):
    total = sum(y_test == k)
    correct = sum((y_test == k) & (y_pred == k))
    print(f"  {letter}: {correct}/{total} = {correct / total:.0%}")

test accuracy: 42/53 = 79.25%
always guessing 'P': 22.64%

recall per class (how much of each class it got right):
  P: 9/12 = 75%
  D: 11/14 = 79%
  G: 10/11 = 91%
  M: 2/4 = 50%
  S: 10/12 = 83%


### Confusion matrix

Each row is the true class, each column is the class the model predicted. The diagonal holds the correct predictions.
Since the classes are ordered, errors near the diagonal (neighboring class) are "less serious" than errors far from it.

In [11]:
print("true \\ predicted " + "".join(f"{letter:>5}" for letter in CLASS_ORDER))
for true, true_letter in enumerate(CLASS_ORDER):
    row = [sum((y_test == true) & (y_pred == p)) for p in range(len(CLASS_ORDER))]
    print(f"{true_letter:>16} " + "".join(f"{n:>5}" for n in row))

true \ predicted     P    D    G    M    S
               P     9    0    3    0    0
               D     3   11    0    0    0
               G     0    0   10    0    1
               M     0    2    0    2    0
               S     0    1    1    0   10


### Classifying a single sample

In [12]:
sample = X_test[0]
predicted_class = predict(model, sample)[0]

print("sample:", sample.astype(int))
print("predicted class:", CLASS_ORDER[predicted_class], "| true class:", y_test_letters[0])

sample: [1 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 1 1 0 0 0 0 1 1 0 0 0 1 0 0]
predicted class: G | true class: G
